# Multi-Agent Research on the Official A2A SDK, with Streaming

This is the second version of my A2A project. The [first notebook](a2a_multi_agent_research.ipynb) implements the A2A protocol by hand to show every message. This one runs **the same three agents and the same LangGraph orchestrator** on Google's official [`a2a-sdk`](https://github.com/a2aproject/a2a-python), and adds the feature the SDK makes easy: **streaming**. While the Researcher works, it streams progress updates ("Searching the web: …") back to the orchestrator as they happen.

| Agent | Built with | A2A skill |
|---|---|---|
| Task Extractor | LangChain (structured output) | `extract_research_brief` |
| Researcher | CrewAI + Tavily | `web_research` (streams a progress update for every search) |
| Writer | Plain OpenAI / Anthropic SDK | `write_cited_answer` |
| Orchestrator | LangGraph, as an A2A client | discovers agents by skill, consumes their event streams |

**What changes from notebook 1:** the protocol layer. The agents, their source and citation checks, and the orchestrator's routing are the same.

**Setup:** add `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`) and `TAVILY_API_KEY` under Colab **Secrets** (🔑) with notebook access on. Steps 1–6 run without any keys.

## Step 1: Install packages

`a2a-sdk` is pinned to an exact version, because its API changed substantially between 0.3 and 1.0. After installing, use **Runtime ▸ Restart session** if Colab asks, then continue from Step 2.

In [1]:
!pip install -q -U "a2a-sdk[http-server]==1.2.2" uvicorn httpx langgraph langchain-core langchain-openai langchain-anthropic "crewai[anthropic]" openai anthropic tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 2: What the SDK takes over

In notebook 1, our own code handled the whole protocol. Here the SDK does that work, and our code adapts the agents to it and consumes the event streams:

| Job | Notebook 1 (by hand) | This notebook (`a2a-sdk`) |
|---|---|---|
| Agent Card and skills | plain dicts | `AgentCard`, `AgentSkill`, `AgentInterface` types |
| Serving the card and JSON-RPC | `http.server` handler | `create_agent_card_routes` + `create_jsonrpc_routes` on a Starlette app, served by uvicorn |
| Creating and storing tasks | our own dict | `DefaultRequestHandler` + `InMemoryTaskStore` |
| Running an agent | call the function, build the Task | an `AgentExecutor` that publishes **events** through a `TaskUpdater` |
| Client | `urllib` + JSON | `A2ACardResolver` + `create_client` |
| Responses | one Task at the end | a **stream** of events: the Task, status updates, artifact updates |

**How an SDK agent works.** An `AgentExecutor` doesn't return a result. It publishes **events** to an event queue, and the SDK delivers them to the client, streamed if the client asked for streaming:

```
executor publishes:  Task (submitted) → status: working → status: working "Searching…" → … → artifact → status: completed
client receives:     the same events, as they happen
```

The SDK enforces the order: a Task event first, then any number of status and artifact updates, ending in a final state (`completed`, `failed`, `input-required`…).

**A2A 1.0 details you'll see in the code:** messages and tasks are **protobuf** objects rather than JSON dicts; task states are enums like `TaskState.TASK_STATE_COMPLETED`; a `Part` holds `text` or `data` directly; and an Agent Card lists its endpoints in `supported_interfaces`.

## Step 3: The server side: one adapter for every agent

`FunctionExecutor` turns any plain function `(text, data, progress) -> AgentReply` into an SDK `AgentExecutor`:

1. It publishes the **Task**, then marks it **working**.
2. It runs the agent in a worker thread, because LangChain, CrewAI and the SDKs are blocking calls and the server is async.
3. The agent can call `progress("…")` at any time. Each call becomes a **streamed status update**, sent back to the server's event loop safely from the worker thread.
4. It publishes the result: an **artifact** and `completed`, or `input-required` / `failed` with the agent's message.

`AgentServer` wraps the SDK's server pieces and runs each agent on its own port in a background thread. Colab already runs an event loop, so each uvicorn server gets its own loop in its own thread.

In [7]:
import asyncio
import socket
import threading
import time
from dataclasses import dataclass

import uvicorn
from starlette.applications import Starlette

from a2a.helpers import (get_data_parts, get_message_text, get_text_parts, new_data_part,
                         new_task_from_user_message, new_text_part)
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill, TaskState


@dataclass
class AgentReply:
    """What an agent function returns. The executor turns it into A2A events."""
    state: str = "completed"          # completed | input-required | failed
    text: str = ""
    data: dict | None = None


def to_python(value):
    """Data parts arrive as protobuf values; convert them back to plain Python."""
    if hasattr(value, "DESCRIPTOR"):
        from google.protobuf.json_format import MessageToDict
        return MessageToDict(value)
    return value


class FunctionExecutor(AgentExecutor):
    """Adapts a plain agent function to the SDK's event-based AgentExecutor interface."""

    def __init__(self, agent_fn):
        self.agent_fn = agent_fn

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # 1. Publish the Task first: the SDK requires it before any update
        task = context.current_task
        if task is None:
            task = new_task_from_user_message(context.message)
            await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue=event_queue, task_id=task.id, context_id=task.context_id)
        await updater.start_work()

        # 2. Read the incoming message: a text part, plus an optional structured data part
        text = get_message_text(context.message)
        data_parts = get_data_parts(context.message.parts)
        data = to_python(data_parts[0]) if data_parts else None

        # 3. progress() is called from the worker thread; each call becomes a streamed status update
        loop = asyncio.get_running_loop()

        def progress(note: str) -> None:
            update = updater.update_status(TaskState.TASK_STATE_WORKING,
                                           message=updater.new_agent_message([new_text_part(note)]))
            asyncio.run_coroutine_threadsafe(update, loop).result(timeout=30)

        # 4. Run the blocking agent in a thread; an exception becomes a failed task
        try:
            reply = await asyncio.to_thread(self.agent_fn, text, data, progress)
        except Exception as err:
            reply = AgentReply(state="failed", text=f"{type(err).__name__}: {err}")

        # 5. Publish the outcome
        parts = [new_text_part(reply.text)] + ([new_data_part(reply.data)] if reply.data is not None else [])
        if reply.state == "completed":
            await updater.add_artifact(parts=parts, name="result")
            await updater.complete()
        elif reply.state == "input-required":
            await updater.requires_input(updater.new_agent_message(parts))
        else:
            await updater.failed(updater.new_agent_message(parts))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Cancel is not supported.")


def _free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def make_skill(skill_id: str, name: str, description: str, tags: list, examples: list = ()) -> AgentSkill:
    return AgentSkill(id=skill_id, name=name, description=description, tags=list(tags), examples=list(examples))


class AgentServer:
    """One agent = Agent Card + request handler + executor, served over JSON-RPC by uvicorn."""

    def __init__(self, name: str, description: str, skill: AgentSkill, agent_fn):
        port = _free_port()
        self.url = f"http://127.0.0.1:{port}"
        self.card = AgentCard(
            name=name, description=description, version="1.0.0",
            default_input_modes=["text/plain", "application/json"],
            default_output_modes=["text/plain", "application/json"],
            capabilities=AgentCapabilities(streaming=True),
            supported_interfaces=[AgentInterface(protocol_binding="JSONRPC", url=self.url, protocol_version="1.0")],
            skills=[skill],
        )
        handler = DefaultRequestHandler(agent_executor=FunctionExecutor(agent_fn),
                                        task_store=InMemoryTaskStore(), agent_card=self.card)
        routes = [*create_agent_card_routes(self.card), *create_jsonrpc_routes(handler, "/")]
        self._server = uvicorn.Server(uvicorn.Config(Starlette(routes=routes), host="127.0.0.1",
                                                     port=port, log_level="warning"))
        self._thread = threading.Thread(target=self._server.run, daemon=True)   # its own event loop

    def start(self):
        self._thread.start()
        deadline = time.time() + 20
        while not self._server.started:
            if time.time() > deadline or not self._thread.is_alive():
                raise RuntimeError(f"{self.card.name} did not start")
            time.sleep(0.05)
        return self

    def stop(self):
        self._server.should_exit = True
        self._thread.join(timeout=5)

## Step 4: The client side: discovery and streamed calls

- `AgentDirectory.register` fetches an Agent Card with the SDK's `A2ACardResolver`, and `find` looks agents up **by skill**.
- `send` opens a **streaming** client, sends one message and consumes the event stream. It keeps the latest task state, collects artifact parts, and passes every progress update to an `on_event` callback so the notebook can show it live.

A response in the stream is a `StreamResponse` with exactly one of `task`, `status_update`, `artifact_update` or `message` set.

In [8]:
import uuid

import httpx
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Message, Role, SendMessageRequest


def state_name(state) -> str:
    """TaskState.TASK_STATE_INPUT_REQUIRED -> 'input-required'"""
    return TaskState.Name(state).removeprefix("TASK_STATE_").lower().replace("_", "-")


@dataclass
class TaskResult:
    agent: str
    state: str
    text: str
    data: dict | None
    events: list          # (kind, state, note) for every event received


async def send(card: AgentCard, text: str, data: dict | None = None, context_id: str | None = None,
               on_event=None, timeout: float = 300.0) -> TaskResult:
    parts = [new_text_part(text)] + ([new_data_part(data)] if data is not None else [])
    message = Message(message_id=str(uuid.uuid4()), role=Role.ROLE_USER, parts=parts,
                      **({"context_id": context_id} if context_id else {}))
    state, output_parts, status_parts, events = TaskState.TASK_STATE_UNSPECIFIED, [], [], []

    async with httpx.AsyncClient(timeout=httpx.Timeout(timeout)) as http:   # research can take minutes
        client = await create_client(agent=card, client_config=ClientConfig(streaming=True, httpx_client=http))
        try:
            async for response in client.send_message(SendMessageRequest(message=message)):
                note = ""
                if response.HasField("task"):                       # a full task snapshot
                    task = response.task
                    state = task.status.state
                    output_parts = [p for artifact in task.artifacts for p in artifact.parts]
                    if task.status.HasField("message"):
                        status_parts = list(task.status.message.parts)
                    kind = "task"
                elif response.HasField("status_update"):            # e.g. working: "Searching the web…"
                    status = response.status_update.status
                    state = status.state
                    if status.HasField("message"):
                        status_parts = list(status.message.parts)
                        note = get_message_text(status.message)
                    kind = "status"
                elif response.HasField("artifact_update"):          # output produced
                    output_parts.extend(response.artifact_update.artifact.parts)
                    kind = "artifact"
                else:                                               # a direct message, without a task
                    status_parts = list(response.message.parts)
                    state, kind = TaskState.TASK_STATE_COMPLETED, "message"
                events.append((kind, state_name(state), note))
                if on_event:
                    on_event(card.name, kind, state_name(state), note)
        finally:
            await client.close()

    final = output_parts if state == TaskState.TASK_STATE_COMPLETED and output_parts else status_parts
    data_parts = get_data_parts(final)
    return TaskResult(agent=card.name, state=state_name(state), text="\n".join(get_text_parts(final)),
                      data=to_python(data_parts[0]) if data_parts else None, events=events)


class AgentDirectory:
    """Knows agents only through their Agent Cards, and finds them by skill ID."""

    def __init__(self):
        self.cards = []

    async def register(self, base_url: str) -> AgentCard:
        async with httpx.AsyncClient(timeout=30) as http:
            card = await A2ACardResolver(httpx_client=http, base_url=base_url).get_agent_card()
        self.cards = [c for c in self.cards if c.name != card.name] + [card]
        return card

    def find(self, skill_id: str) -> AgentCard:
        for card in self.cards:
            if any(s.id == skill_id for s in card.skills):
                return card
        raise LookupError(f"no registered agent offers the skill '{skill_id}'")

    def show(self):
        for card in self.cards:
            print(f"  {card.name:<16} {card.supported_interfaces[0].url:<24} skills: {', '.join(s.id for s in card.skills)}")

## Step 5: The orchestrator (LangGraph, async)

The same graph as notebook 1, with async nodes because the SDK client is async:

```
START → extract ──completed──→ research ──completed──→ write → END
           │                       │
           ├─ input-required ──→ END (ask the user the agent's question)
           └─ failed ──────────→ END (report which agent failed and why)
```

Progress updates from any agent are printed as they arrive, and counted in the trace.

In [9]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class ResearchState(TypedDict, total=False):
    question: str
    context_id: str
    brief: dict
    findings: list
    answer: str
    sources: list
    status: str                                   # set when the run stops: completed | input-required | failed
    message: str
    trace: Annotated[list, operator.add]


def print_event(agent: str, kind: str, state: str, note: str) -> None:
    if note:
        print(f"    ↳ {agent}: {note}")


def build_orchestrator(directory: AgentDirectory, on_event=print_event):
    async def call(skill_id: str, text: str, data: dict | None, state: ResearchState):
        card = directory.find(skill_id)
        start = time.perf_counter()
        try:
            result = await send(card, text, data, context_id=state["context_id"], on_event=on_event)
        except Exception as err:                  # unreachable agent or protocol error
            result = TaskResult(card.name, "failed", f"{type(err).__name__}: {err}", None, [])
        entry = {"agent": card.name, "skill": skill_id, "state": result.state, "events": len(result.events),
                 "seconds": round(time.perf_counter() - start, 1)}
        return result, entry

    def stopped(result: TaskResult, entry: dict) -> dict | None:
        if result.state == "completed":
            return None
        return {"status": result.state, "message": f"{result.agent}: {result.text}", "trace": [entry]}

    async def extract(state: ResearchState) -> dict:
        result, entry = await call("extract_research_brief", state["question"], None, state)
        return stopped(result, entry) or {"brief": result.data, "trace": [entry]}

    async def research(state: ResearchState) -> dict:
        brief = state["brief"]
        result, entry = await call("web_research", brief["task"], brief, state)
        return stopped(result, entry) or {"findings": result.data["findings"], "trace": [entry]}

    async def write(state: ResearchState) -> dict:
        payload = {"task": state["brief"]["task"], "audience": state["brief"].get("audience", "general"),
                   "findings": state["findings"]}
        result, entry = await call("write_cited_answer", state["brief"]["task"], payload, state)
        data = result.data or {}
        return stopped(result, entry) or {"answer": data.get("answer", result.text), "sources": data.get("sources", []),
                                          "status": "completed", "trace": [entry]}

    def route(next_node: str):
        return lambda state: END if state.get("status") else next_node

    graph = StateGraph(ResearchState)
    graph.add_node("extract", extract)
    graph.add_node("research", research)
    graph.add_node("write", write)
    graph.add_edge(START, "extract")
    graph.add_conditional_edges("extract", route("research"), ["research", END])
    graph.add_conditional_edges("research", route("write"), ["write", END])
    graph.add_edge("write", END)
    return graph.compile()


async def run(orchestrator, question: str, show: bool = True) -> dict:
    result = await orchestrator.ainvoke({"question": question, "context_id": str(uuid.uuid4()), "trace": []})
    if show:
        print("\nA2A calls:")
        for t in result["trace"]:
            print(f"  {t['agent']:<16} {t['skill']:<24} → {t['state']:<15} {t['events']:>3} events {t['seconds']:>7.1f} s")
        print(f"\nStatus: {result.get('status')}")
        if result.get("status") == "completed":
            print("\n" + result["answer"])
            print("\nSources:")
            for i, url in enumerate(result["sources"], 1):
                print(f"  [{i}] {url}")
        else:
            print(result.get("message"))
    return result

## Step 6: Offline tests with stub agents (no API keys)

Stub agents with the same skills run on real SDK servers. The stub Researcher calls `progress()` twice, so the tests can check that **streamed updates** arrive before the result.

In [10]:
for s in globals().get("SERVERS", []):
    s.stop()
SERVERS = []

EXTRACT_SKILL = make_skill("extract_research_brief", "Extract research brief",
    "Turns a user request into a research task with web search queries, or asks a clarifying question.",
    ["planning", "intent"], ["Summarize the latest GenAI trends for enterprise"])
RESEARCH_SKILL = make_skill("web_research", "Web research",
    "Searches the web and returns findings, each tied to the URL it came from. Streams progress.",
    ["research", "web-search"], ["Recent enterprise GenAI adoption trends"])
WRITE_SKILL = make_skill("write_cited_answer", "Write cited answer",
    "Writes a concise answer for a given audience, citing only the findings it is given.",
    ["writing", "synthesis"])


def stub_extractor(text, data, progress):
    if len(text.split()) < 3:
        return AgentReply(state="input-required", text="What topic should I research?")
    return AgentReply(text="Research: " + text, data={"task": text, "search_queries": [text], "audience": "general"})


def stub_researcher(text, data, progress):
    if "crash" in text:
        raise RuntimeError("search service unavailable")
    progress("Searching the web: query 1")
    progress("Searching the web: query 2")
    return AgentReply(text="2 findings", data={"findings": [
        {"point": "Finding one", "source_url": "https://example.com/a"},
        {"point": "Finding two", "source_url": "https://example.com/b"}]})


def stub_writer(text, data, progress):
    urls = [f["source_url"] for f in data["findings"]]
    return AgentReply(text="Answer", data={"answer": "Finding one [1]. Finding two [2].", "sources": urls})


stubs = [AgentServer("Stub Extractor", "Test double", EXTRACT_SKILL, stub_extractor).start(),
         AgentServer("Stub Researcher", "Test double", RESEARCH_SKILL, stub_researcher).start(),
         AgentServer("Stub Writer", "Test double", WRITE_SKILL, stub_writer).start()]
SERVERS += stubs

# 1. Discovery through the SDK's card resolver
stub_directory = AgentDirectory()
for s in stubs:
    await stub_directory.register(s.url)
assert stub_directory.find("web_research").name == "Stub Researcher"
print("✓ 1. discovery: Agent Cards fetched with A2ACardResolver and looked up by skill")

# 2. A completed task: text and structured data survive the round trip through protobuf
r = await send(stub_directory.find("extract_research_brief"), "GenAI trends for enterprise")
assert r.state == "completed" and r.data["task"] == "GenAI trends for enterprise", r
print("✓ 2. message: completed task with text and data parts")

# 3. Streaming: progress updates arrive as status events before the final result
seen = []
r = await send(stub_directory.find("web_research"), "trends", {"task": "trends", "search_queries": ["x"]},
               on_event=lambda agent, kind, state, note: seen.append(note))
assert r.state == "completed" and len(r.data["findings"]) == 2
assert [n for n in seen if n.startswith("Searching")] == ["Searching the web: query 1", "Searching the web: query 2"], seen
print(f"✓ 3. streaming: {len(r.events)} events received, including both progress updates in order")

# 4. Task states: input-required carries the question; an exception becomes a failed task
r = await send(stub_directory.find("extract_research_brief"), "trends", timeout=20)
assert r.state == "input-required" and r.text == "What topic should I research?", r
r = await send(stub_directory.find("web_research"), "crash please")
assert r.state == "failed" and "search service unavailable" in r.text, r
try:
    stub_directory.find("translate")
    raise AssertionError("expected an error for an unknown skill")
except LookupError:
    pass
print("✓ 4. task states: input-required and failed are reported; unknown skills are rejected")

# 5. The orchestrator end to end, on all three paths
orchestrator = build_orchestrator(stub_directory, on_event=None)
done = await run(orchestrator, "Summarize GenAI trends for enterprise", show=False)
assert done["status"] == "completed" and done["answer"] == "Finding one [1]. Finding two [2]."
assert [t["skill"] for t in done["trace"]] == ["extract_research_brief", "web_research", "write_cited_answer"]
asked = await run(orchestrator, "trends", show=False)
assert asked["status"] == "input-required" and len(asked["trace"]) == 1
failed = await run(orchestrator, "please crash the research step", show=False)
assert failed["status"] == "failed" and "Stub Researcher" in failed["message"]
print("✓ 5. orchestrator: completes, stops for clarification, and reports a failing agent by name")
print("\nAll offline tests passed.")

✓ 1. discovery: Agent Cards fetched with A2ACardResolver and looked up by skill
✓ 2. message: completed task with text and data parts
✓ 3. streaming: 6 events received, including both progress updates in order
✓ 4. task states: input-required and failed are reported; unknown skills are rejected
✓ 5. orchestrator: completes, stops for clarification, and reports a failing agent by name

All offline tests passed.


## Step 7: Connect to the LLM and web search

Set `PROVIDER` to `"openai"` or `"anthropic"`.

In [11]:
import os

def _load(key):
    """Colab Secrets first, then an environment variable."""
    try:
        from google.colab import userdata
        value = userdata.get(key)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(key)


PROVIDER = "openai"   # or "anthropic"
MODEL = {"openai": "gpt-5-mini", "anthropic": "claude-sonnet-4-5"}[PROVIDER]
KEY_NAME = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[PROVIDER]

for key in (KEY_NAME, "TAVILY_API_KEY"):
    value = _load(key)
    if not value:
        raise RuntimeError(f"Add {key} to Colab Secrets (🔑) and turn on notebook access.")
    os.environ[key] = value       # LangChain, CrewAI and the SDKs all read keys from the environment

os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"   # no trace upload, and no interactive "view traces?" prompt
os.environ["OTEL_SDK_DISABLED"] = "true"
print(f"Using {PROVIDER} / {MODEL}")

Using openai / gpt-5-mini


## Step 8: The three real agents

The same agents as notebook 1, with one addition: each takes a `progress` callback. The Researcher reports every web search, so the orchestrator can show what it's doing during its longest step.

**Agent 1: Task Extractor (LangChain).** Returns a validated research brief, or ends as `input-required` with a clarifying question.

In [12]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class ResearchBrief(BaseModel):
    task: str = Field(description="The user's request restated as one clear research task")
    search_queries: list[str] = Field(description="2 or 3 focused web search queries")
    audience: str = Field(description="Who the answer is for, e.g. 'enterprise leaders', or 'general' if not stated")
    needs_clarification: bool = Field(description="True only if the request is too vague to research at all")
    clarifying_question: str = Field(description="The question to ask the user if needs_clarification is true, else an empty string")


def make_chat_model():
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL)
    from langchain_anthropic import ChatAnthropic
    return ChatAnthropic(model=MODEL, max_tokens=2000)


extract_chain = ChatPromptTemplate.from_messages([
    ("system", "You turn a user's request into a research brief for a web researcher. "
               "Ask for clarification only if the request has no identifiable topic."),
    ("human", "{request}"),
]) | make_chat_model().with_structured_output(ResearchBrief)


def task_extractor(text: str, data: dict | None, progress) -> AgentReply:
    progress("Turning the request into a research brief")
    brief = extract_chain.invoke({"request": text})
    if brief.needs_clarification:
        return AgentReply(state="input-required", text=brief.clarifying_question or "Could you say more about what you need?")
    brief.search_queries = [q for q in brief.search_queries if q.strip()][:3] or [brief.task]
    return AgentReply(text=brief.task, data=brief.model_dump())

**Agent 2: Researcher (CrewAI + Tavily).** Streams a progress update for every search, then checks that every cited URL came from a real search result.

In [13]:
from typing import Any

from crewai import LLM, Agent, Crew, Task
from crewai.tools import BaseTool
from tavily import TavilyClient


class Finding(BaseModel):
    point: str = Field(description="One specific, factual finding")
    source_url: str = Field(description="The URL of the search result this finding came from")


class ResearchFindings(BaseModel):
    findings: list[Finding]


class SearchInput(BaseModel):
    query: str = Field(description="A web search query")


class WebSearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for current information. Returns titles, URLs and snippets."
    args_schema: type[BaseModel] = SearchInput
    seen_urls: list = Field(default_factory=list)      # every URL this tool returned, for the source check
    progress: Any = None                               # streams each search back to the client

    def _run(self, query: str) -> str:
        if self.progress:
            self.progress(f"Searching the web: {query}")
        results = TavilyClient(api_key=os.environ["TAVILY_API_KEY"]).search(query, max_results=5)["results"]
        self.seen_urls.extend(r["url"] for r in results)
        return "\n\n".join(f"{r['title']}\nURL: {r['url']}\n{r['content'][:600]}" for r in results)


def _norm(url: str) -> str:
    return url.strip().rstrip("/").lower()


def researcher(text: str, data: dict | None, progress) -> AgentReply:
    brief = data or {"task": text, "search_queries": [text]}
    search = WebSearchTool(progress=progress)
    agent = Agent(role="Web Researcher",
                  goal="Find current, specific, well-sourced information",
                  backstory="You research with a search tool and never state anything you did not find in a result.",
                  tools=[search], llm=LLM(model=f"{PROVIDER}/{MODEL}"), allow_delegation=False, verbose=False)
    job = Task(description=(f"Research this task: {brief['task']}\n"
                            f"Start with these searches: {brief['search_queries']}\n"
                            "Return 5 findings. Each source_url must be copied exactly from a search result."),
               expected_output="5 findings, each with the URL it came from", agent=agent,
               output_pydantic=ResearchFindings)
    result = Crew(agents=[agent], tasks=[job], verbose=False, tracing=False).kickoff()
    findings = result.pydantic.findings if result.pydantic else ResearchFindings.model_validate_json(result.raw).findings

    # Source check, in code: keep only findings whose URL the search tool really returned
    progress(f"Checking {len(findings)} findings against {len(search.seen_urls)} search results")
    seen = {_norm(u) for u in search.seen_urls}
    kept = [f.model_dump() for f in findings if _norm(f.source_url) in seen]
    dropped = len(findings) - len(kept)
    if not kept:
        return AgentReply(state="failed", text=f"None of the {len(findings)} findings could be traced to a search result.")
    return AgentReply(text=f"{len(kept)} sourced findings ({dropped} dropped: URL not in search results)",
                      data={"findings": kept, "dropped": dropped, "searched_urls": len(seen)})

**Agent 3: Writer (plain SDK, no framework).** Writes the answer with numbered citations; code checks every citation and allows one retry.

In [14]:
import re

CITATION_RE = re.compile(r"\[(\d+)\]")


def call_llm(prompt: str) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        reply = OpenAI().chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}])
        return reply.choices[0].message.content or ""
    import anthropic
    reply = anthropic.Anthropic().messages.create(model=MODEL, max_tokens=2000,
                                                  messages=[{"role": "user", "content": prompt}])
    return "".join(block.text for block in reply.content if block.type == "text")


def citation_problems(answer: str, n_sources: int) -> list:
    cited = {int(n) for n in CITATION_RE.findall(answer)}
    problems = [f"[{n}] is not a source number (valid: 1–{n_sources})" for n in sorted(cited) if not 1 <= n <= n_sources]
    if not cited:
        problems.append("the answer cites no sources")
    return problems


def writer(text: str, data: dict | None, progress) -> AgentReply:
    findings = data["findings"]
    sources = list(dict.fromkeys(f["source_url"] for f in findings))          # unique, in order
    numbered = "\n".join(f"- {f['point']} [{sources.index(f['source_url']) + 1}]" for f in findings)
    prompt = (f"Task: {data['task']}\nAudience: {data.get('audience', 'general')}\n\n"
              f"Findings, with source numbers:\n{numbered}\n\n"
              "Write a concise answer (one short paragraph plus up to 5 bullet points) using ONLY these findings. "
              "Cite source numbers in square brackets, e.g. [1]. Do not add facts that are not in the findings.")
    progress(f"Writing from {len(findings)} findings and {len(sources)} sources")
    answer = call_llm(prompt)
    problems = citation_problems(answer, len(sources))
    if problems:                                                             # one retry with the exact problem
        progress("Fixing citations: " + "; ".join(problems))
        answer = call_llm(prompt + f"\n\nYour previous draft had problems: {'; '.join(problems)}. Rewrite it.")
        problems = citation_problems(answer, len(sources))
    if problems:
        return AgentReply(state="failed", text="Citation check failed: " + "; ".join(problems))
    return AgentReply(text=answer, data={"answer": answer, "sources": sources})


print("Agents defined: task_extractor (LangChain), researcher (CrewAI), writer (plain SDK)")

Agents defined: task_extractor (LangChain), researcher (CrewAI), writer (plain SDK)


## Step 9: Start the agents and discover them

The stubs are stopped, the real agents start on SDK servers, and the orchestrator is built by the same `build_orchestrator` the tests used.

In [15]:
import json

from google.protobuf.json_format import MessageToDict

for s in SERVERS:
    s.stop()
SERVERS = [
    AgentServer("Task Extractor", "LangChain agent that turns requests into research briefs", EXTRACT_SKILL, task_extractor).start(),
    AgentServer("Researcher", "CrewAI agent that researches the web with source-checked findings", RESEARCH_SKILL, researcher).start(),
    AgentServer("Writer", "Framework-free agent that writes cited answers", WRITE_SKILL, writer).start(),
]

directory = AgentDirectory()
for s in SERVERS:
    await directory.register(s.url)
print("Registered agents:")
directory.show()

orchestrator = build_orchestrator(directory)

print("\nThe Researcher's Agent Card (A2A 1.0 format):")
print(json.dumps(MessageToDict(directory.find("web_research")), indent=2))

Registered agents:
  Task Extractor   http://127.0.0.1:54781   skills: extract_research_brief
  Researcher       http://127.0.0.1:45819   skills: web_research
  Writer           http://127.0.0.1:40075   skills: write_cited_answer

The Researcher's Agent Card (A2A 1.0 format):
{
  "name": "Researcher",
  "description": "CrewAI agent that researches the web with source-checked findings",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:45819",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain",
    "application/json"
  ],
  "defaultOutputModes": [
    "text/plain",
    "application/json"
  ],
  "skills": [
    {
      "id": "web_research",
      "name": "Web research",
      "description": "Searches the web and returns findings, each tied to the URL it came from. Streams progress.",
      "tags": [
        "research",
        "web-search

## Step 10: Run a research question, with live progress

The `↳` lines are streamed status updates, printed the moment each agent sends them.

In [16]:
result = await run(orchestrator, "Can you summarize the latest trends in GenAI for enterprise?")

    ↳ Task Extractor: Turning the request into a research brief
    ↳ Researcher: Searching the web: 2026 enterprise generative AI trends adoption use cases Gartner McKinsey Forrester report 2025 2026
    ↳ Researcher: Searching the web: NIST generative AI guidance 2025 2026 hallucination risk management
    ↳ Researcher: Searching the web: Gartner 2026 generative AI adoption enterprise 2026 report 'generative AI' adoption 2025 2026 Gartner CIO Agenda 2026
    ↳ Researcher: Searching the web: McKinsey The State of AI 2026 report generative AI adoption 2026
    ↳ Researcher: Searching the web: NIST AI RMF generative AI guidance 2025 2026 site:nist.gov
    ↳ Researcher: Searching the web: EU AI Act latest 2024 2025 2026 European Commission 'AI Act' official text 2024 2025 2026


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

    ↳ Researcher: Checking 5 findings against 30 search results
Would you like to view your execution traces? [y/N] (20s timeout):     ↳ Writer: Writing from 5 findings and 5 sources
N


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


A2A calls:
  Task Extractor   extract_research_brief   → completed         5 events    11.2 s
  Researcher       web_research             → completed        11 events   165.4 s
  Writer           write_cited_answer       → completed         5 events     8.1 s

Status: completed

Enterprise adoption of generative AI is accelerating: nearly nine in ten respondents report regular AI use in at least one business function, and 44% now say AI is scaling across their enterprise (up from 38% a year ago) [1]. At the same time regulatory and standards activity is intensifying (the EU AI Act entered into force in August 2024 with staged transparency and model obligations through August 2026) [3], standards bodies are issuing guidance (NIST released a concept note for an AI RMF Profile on Trustworthy AI in Critical Infrastructure on April 7, 2026) [2], and analyst and consulting firms see major architectural and ROI implications for enterprises (Gartner forecasts widespread deployment of task‑spe

## Step 11: The Researcher's event stream

Every event the orchestrator received from the Researcher, in order. In notebook 1 this step was a single response at the end.

In [17]:
card = directory.find("web_research")
events = []
brief = {"task": "Recent trends in AI for sports analytics", "search_queries": ["AI sports analytics trends 2026"],
         "audience": "general"}
r = await send(card, brief["task"], brief, on_event=lambda agent, kind, state, note: events.append((time.perf_counter(), kind, state, note)))
start = events[0][0]
for t, kind, state, note in events:
    print(f"  {t - start:>6.1f} s  {kind:<9} {state:<15} {note}")
print(f"\nFinal state: {r.state}, {len(r.data['findings']) if r.data else 0} findings")

     0.0 s  task      submitted       
     0.0 s  status    working         
    19.8 s  status    working         Searching the web: AI sports analytics trends 2026
    59.4 s  status    working         Checking 5 findings against 5 search results
    59.4 s  artifact  working         
    59.4 s  status    completed       

Final state: completed, 5 findings


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

## Step 12: A vague request: the `input-required` state

With no identifiable topic, the Task Extractor should end its task as `input-required`, and the orchestrator returns the question instead of researching a guess. (The decision is the LLM's, so a capable model may still find a topic.)

In [18]:
vague = await run(orchestrator, "Can you look into it for me?")

    ↳ Task Extractor: Turning the request into a research brief
    ↳ Task Extractor: What specifically would you like me to look into? Please provide: 1) the topic or item (what does “it” refer to), 2) the goal or questions you want answered, 3) desired deliverable (summary, full report, links, steps to resolve), 4) deadline or urgency, and 5) any relevant links, documents, or constraints.

A2A calls:
  Task Extractor   extract_research_brief   → input-required    4 events    11.0 s

Status: input-required
Task Extractor: What specifically would you like me to look into? Please provide: 1) the topic or item (what does “it” refer to), 2) the goal or questions you want answered, 3) desired deliverable (summary, full report, links, steps to resolve), 4) deadline or urgency, and 5) any relevant links, documents, or constraints.


## Notes

- **Notebook 1 vs. this notebook:** the protocol-side code is about the same length in both (roughly 170 lines without comments), but it does different work. In notebook 1 it implemented the protocol; here the SDK implements it, and the code instead bridges blocking agents to an async server, streams progress, and consumes event streams. The agents and the orchestrator's logic didn't change, which is the point of building on a protocol.
- **What streaming adds:** the Researcher's step takes a minute or more. With `message/send` the orchestrator saw nothing until it ended; with streaming it sees every search as it happens, and could show it in a UI or time out a stuck agent.
- **Version pin:** the notebook is pinned to `a2a-sdk==1.2.2`. Version 1.0 moved to protobuf types and changed the server and client APIs, so code written for 0.3 tutorials won't run on it.
- **Not covered:** push notifications, authenticated extended Agent Cards, and the gRPC and REST transports, which the SDK also supports.